In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
import os
import io
import glob
import struct
from multiprocessing import Pool, cpu_count
from PIL import Image
from tqdm import tqdm

from tensorflow.core.example.example_pb2 import Example


# ---- TFRecord Reader ----
def read_tfrecord(file_path):
    with open(file_path, "rb") as f:
        while True:
            length_bytes = f.read(8)
            if not length_bytes:
                break

            length = struct.unpack("<Q", length_bytes)[0]
            f.read(4)  # crc
            data = f.read(length)
            f.read(4)  # crc

            yield data


# ---- Parser ----
def parse_example(serialized):
    example = Example()
    example.ParseFromString(serialized)

    features = example.features.feature

    image = features["image"].bytes_list.value[0]
    label = features["class"].int64_list.value[0]
    img_id = features["id"].bytes_list.value[0].decode()

    return image, label, img_id


# ---- Worker ----
def process_file(file_path):
    local_count = 0

    for record in read_tfrecord(file_path):
        img_bytes, label, img_id = parse_example(record)

        class_dir = os.path.join(OUTPUT_PATH, str(label))
        os.makedirs(class_dir, exist_ok=True)

        img = Image.open(io.BytesIO(img_bytes))
        img.save(os.path.join(class_dir, f"{img_id}.jpg"))

        local_count += 1

    return local_count

2026-05-03 12:11:20.876864: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777810281.189852      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777810281.290447      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777810282.051890      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777810282.051936      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777810282.051939      57 computation_placer.cc:177] computation placer alr

In [3]:
BASE_PATH = "/kaggle/input/competitions/tpu-getting-started"

In [4]:
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader

IMAGE_SIZE = 224

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [5]:
import timm
import torch
import torch.nn as nn

NUM_CLASSES = 104  # flower classes

def build_model(model_name="convnext_tiny", pretrained=True):
    model = timm.create_model(
        model_name,
        pretrained=pretrained,
        num_classes=NUM_CLASSES
    )
    return model

In [6]:
import cv2
import numpy as np
import os

TEST_OUTPUT_PATH = "/kaggle/output/test_images"
os.makedirs(TEST_OUTPUT_PATH, exist_ok=True)

from tensorflow.core.example.example_pb2 import Example

def parse_test_example(serialized):
    example = Example()
    example.ParseFromString(serialized)

    features = example.features.feature

    image = features["image"].bytes_list.value[0]
    img_id = features["id"].bytes_list.value[0].decode()

    return image, img_id

def process_test_file(file_path):
    count = 0

    for record in read_tfrecord(file_path):
        img_bytes, img_id = parse_test_example(record)

        # fast decode
        img_array = np.frombuffer(img_bytes, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

        cv2.imwrite(os.path.join(TEST_OUTPUT_PATH, f"{img_id}.jpg"), img)

        count += 1

    return count

In [7]:
TEST_INPUT_PATH = BASE_PATH + "/tfrecords-jpeg-224x224/test"

files_test = glob.glob(TEST_INPUT_PATH + "/*.tfrec")

num_workers = min(4, cpu_count())  # safer on Kaggle

with Pool(num_workers) as p:
    results = list(tqdm(p.imap(process_test_file, files_test), total=len(files_test)))

print(f"Total test images processed: {sum(results)}")

100%|██████████| 16/16 [00:02<00:00,  6.48it/s]

Total test images processed: 7382


In [8]:
from torch.utils.data import Dataset
from PIL import Image
import os

class TestDataset(Dataset):
    def __init__(self, folder, transform=None):
        self.paths = sorted([
            os.path.join(folder, f)
            for f in os.listdir(folder)
            if f.endswith(".jpg")
        ])
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        img_id = os.path.basename(path).replace(".jpg", "")
        return img, img_id

In [9]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

test_dataset = TestDataset(
    "/kaggle/output/test_images",
    transform=transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_model(model_name="efficientnet_b0", pretrained=False);

model.load_state_dict(torch.load(
    "/kaggle/input/datasets/sadiq18/petalstothemetalmodels/best_model_efficient_b0_ema.pth",
    map_location=device
));

model.to(device);

In [11]:
from tqdm import tqdm

In [12]:
ids = []
preds = []

model.eval()
with torch.no_grad():
    for images, img_ids in tqdm(test_loader):
        images = images.to(device)

        outputs = model(images)
        pred = outputs.argmax(dim=1).cpu().numpy()

        ids.extend(img_ids)
        preds.extend(pred)

100%|██████████| 116/116 [05:29<00:00,  2.84s/it]


In [13]:
import pandas as pd

df = pd.DataFrame({
    "id": ids,
    "label": preds
})

df.to_csv("submission.csv", index=False)

In [14]:
df.head()

,id,label
0,001e13533,40
1,0021f0d33,53
2,003882deb,93
3,0039e54f0,42
4,003b89961,48
